In [ ]:
# ==========================================
# Setup and Imports
# ==========================================

import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw
Image.MAX_IMAGE_PIXELS = None

import os
import numpy as np
from tqdm.notebook import tqdm
import ipywidgets as widgets
from IPython.display import display, clear_output

WINDOW_SIZE = 50


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Working on: {DEVICE}")

Working on: cuda


In [ ]:
# ==========================================
# Model Architecture
# ==========================================

class ArtifactCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # -> 24x24
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # -> 12x12
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2) # -> 6x6
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 6 * 6, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

In [ ]:
# ==========================================
# Sliding Window Dataset
# ==========================================

class SlidingWindowDataset(Dataset):
    def __init__(self, pil_image, window_size=50, stride=10, transform=None):
        self.image = pil_image
        self.window_size = window_size
        self.transform = transform
        self.coords = []
        w, h = pil_image.size
        for y in range(0, h - window_size + 1, stride):
            for x in range(0, w - window_size + 1, stride):
                self.coords.append((x, y, x + window_size, y + window_size))

    def __len__(self): return len(self.coords)

    def __getitem__(self, idx):
        box = self.coords[idx]
        window = self.image.crop(box)
        if self.transform: window = self.transform(window)
        return window, torch.tensor(box)

In [ ]:
# ==========================================
# Inference Engine
# ==========================================

def get_all_predictions(image_path, model, batch_size=512):
    img = Image.open(image_path).convert("RGB")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])

    dataset = SlidingWindowDataset(img, window_size=50, stride=10, transform=transform)
    loader = DataLoader(dataset, batch_size=batch_size, num_workers=0, pin_memory=True)

    model.eval()
    results = {"boxes": [], "scores": []}

    print(f"🔍 Analyzing {len(dataset)} patches with Binary Model...")
    with torch.no_grad():
       with torch.amp.autocast('cuda'):
            for windows, boxes in tqdm(loader):
                outputs = model(windows.to(DEVICE))
                probs = torch.softmax(outputs, dim=1)

                distorted_scores = probs[:, 1]

                results["boxes"].append(boxes)
                results["scores"].append(distorted_scores.cpu())

    results["boxes"] = torch.cat(results["boxes"]).float()
    results["scores"] = torch.cat(results["scores"])
    return img, results

In [ ]:
# ==========================================
#  Interactive UI & NMS
# ==========================================

def display_interactive_results(original_img, results):
    def update_view(threshold, nms_iou):
        mask = results["scores"] > threshold
        if not mask.any():
            print("No distortions detected above threshold.")
            display(original_img.resize((800, 600)))
            return

        f_boxes = results["boxes"][mask]
        f_scores = results["scores"][mask]

        margin = 20
        inflated_boxes = f_boxes.clone()
        inflated_boxes[:, 0] -= margin
        inflated_boxes[:, 1] -= margin
        inflated_boxes[:, 2] += margin
        inflated_boxes[:, 3] += margin

        keep = torchvision.ops.nms(inflated_boxes, f_scores, nms_iou)

        final_boxes = f_boxes[keep]
        final_scores = f_scores[keep]

        draw_img = original_img.copy()
        draw = ImageDraw.Draw(draw_img)
        for box, score in zip(final_boxes, final_scores):
            draw.rectangle(list(box), outline="red", width=4)

            center_x = (box[0] + box[2]) / 2
            center_y = (box[1] + box[3]) / 2
            draw.ellipse([center_x-3, center_y-3, center_x+3, center_y+3], fill="red")

            draw.text((box[0], box[1]-15), f"Distorted {score:.2f}", fill="red")

        print(f"✅ Found {len(final_boxes)} suspected points (Local Maxima)")

        display(draw_img.resize((1000, 750)))

    widgets.interact(update_view,
                     threshold=widgets.FloatSlider(value=0.90, min=0.1, max=0.99, step=0.05),
                     nms_iou=widgets.FloatSlider(value=0.10, min=0.05, max=0.5, step=0.05))

In [ ]:
# ==========================================
# Heatmap
# ==========================================
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from PIL import Image

def display_interactive_heatmap(original_img, results):
    w, h = original_img.size
    original_bgr = cv2.cvtColor(np.array(original_img), cv2.COLOR_RGB2BGR)

    def update_heatmap(threshold, alpha):
        mask = results["scores"] > threshold
        if not mask.any():
            print("No distortions detected above threshold.")
            display(original_img.resize((800, 600)))
            return

        f_boxes = results["boxes"][mask].cpu().numpy()
        f_scores = results["scores"][mask].cpu().numpy()

        score_map = np.zeros((h, w), dtype=np.float32)
        count_map = np.zeros((h, w), dtype=np.float32)

        for box, score in zip(f_boxes, f_scores):
            x1, y1, x2, y2 = box.astype(int)
            score_map[y1:y2, x1:x2] += score
            count_map[y1:y2, x1:x2] += 1

        empty_mask = (count_map == 0)
        count_map[empty_mask] = 1
        avg_map = score_map / count_map

        heatmap_norm = (avg_map * 255).astype(np.uint8)
        colored_heatmap = cv2.applyColorMap(heatmap_norm, cv2.COLORMAP_JET)
        blended = cv2.addWeighted(original_bgr, 1 - alpha, colored_heatmap, alpha, 0)

        blended[empty_mask] = original_bgr[empty_mask]

        final_rgb = cv2.cvtColor(blended, cv2.COLOR_BGR2RGB)
        final_pil = Image.fromarray(final_rgb)

        print(f"🔥 Showing heatmap for {len(f_boxes)} patches above threshold {threshold:.2f}")
        display(final_pil.resize((1000, 750)))

    widgets.interact(update_heatmap,
                     threshold=widgets.FloatSlider(value=0.90, min=0.1, max=0.99, step=0.05),
                     alpha=widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.1, description='Opacity'))

In [ ]:
# ==========================================
#  Execution
# ==========================================
import time
import os

MODEL_PATH = "best_binary_model.pth"
IMAGE_PATH = "/content/TE42@gt.bmp"

if not os.path.exists(MODEL_PATH):
    print(f"Error: Model file {MODEL_PATH} not found. Make sure you uploaded it to Colab.")
else:
    model = ArtifactCNN().to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print("Model loaded successfully!")

    print("Starting Inference...")
    start_time = time.time()

    original_img, raw_results = get_all_predictions(IMAGE_PATH, model, batch_size=512)

    end_time = time.time()
    print(f"Inference completed in {end_time - start_time:.2f} seconds.")

    print("\n--- Bounding Boxes (NMS) ---")
    display_interactive_results(original_img, raw_results)

    print("\n--- Heatmap View ---")
    display_interactive_heatmap(original_img, raw_results)

In [ ]:
# ==========================================
# Output Requirement
# ==========================================
print("📝 Top 20 Suspected Centers (Local Maxima):")

mask = raw_results["scores"] > 0.90
f_boxes = raw_results["boxes"][mask]
f_scores = raw_results["scores"][mask]

margin = 20
inflated_boxes = f_boxes.clone()
inflated_boxes[:, 0] -= margin; inflated_boxes[:, 1] -= margin
inflated_boxes[:, 2] += margin; inflated_boxes[:, 3] += margin

keep = torchvision.ops.nms(inflated_boxes, f_scores, iou_threshold=0.1)
final_boxes = f_boxes[keep]
final_scores = f_scores[keep]

limit = min(20, len(final_boxes))
for idx in range(limit):
    box = final_boxes[idx]
    score = final_scores[idx]
    center_x = (box[0] + box[2]) / 2
    center_y = (box[1] + box[3]) / 2
    print(f"{idx+1}. Center: ({center_x:.1f}, {center_y:.1f}) | Score: {score:.4f}")

📝 Top 20 Suspected Centers (Local Maxima):
1. Center: (1815.0, 705.0) | Score: 1.0000
2. Center: (6025.0, 735.0) | Score: 1.0000
3. Center: (10255.0, 735.0) | Score: 1.0000
4. Center: (15035.0, 825.0) | Score: 1.0000
5. Center: (14365.0, 835.0) | Score: 1.0000
6. Center: (15435.0, 835.0) | Score: 1.0000
7. Center: (14615.0, 845.0) | Score: 1.0000
8. Center: (12585.0, 855.0) | Score: 1.0000
9. Center: (16175.0, 1145.0) | Score: 1.0000
10. Center: (155.0, 1155.0) | Score: 1.0000
11. Center: (16165.0, 1975.0) | Score: 1.0000
12. Center: (8215.0, 2205.0) | Score: 1.0000
13. Center: (16175.0, 2375.0) | Score: 1.0000
14. Center: (7765.0, 2615.0) | Score: 1.0000
15. Center: (7725.0, 2755.0) | Score: 1.0000
16. Center: (7555.0, 2765.0) | Score: 1.0000
17. Center: (8415.0, 2785.0) | Score: 1.0000
18. Center: (8685.0, 2785.0) | Score: 1.0000
19. Center: (7945.0, 2795.0) | Score: 1.0000
20. Center: (8075.0, 2805.0) | Score: 1.0000


In [ ]:
import json

def export_suspects_to_json(raw_results, threshold=0.90, nms_iou=0.2,
                             margin=20,
                             output_file="suspected_patches.json"):
    mask = raw_results["scores"] > threshold
    f_boxes = raw_results["boxes"][mask]
    f_scores = raw_results["scores"][mask]

    inflated_boxes = f_boxes.clone()
    inflated_boxes[:, 0] -= margin
    inflated_boxes[:, 1] -= margin
    inflated_boxes[:, 2] += margin
    inflated_boxes[:, 3] += margin

    keep = torchvision.ops.nms(inflated_boxes, f_scores, nms_iou)

    final_boxes = f_boxes[keep].cpu().numpy().tolist()
    final_scores = f_scores[keep].cpu().numpy().tolist()

    suspects_list = []
    for box, score in zip(final_boxes, final_scores):
        suspects_list.append({
            "box": box,
            "binary_score": round(score, 4)
        })

    with open(output_file, 'w') as f:
        json.dump(suspects_list, f, indent=4)

    print(f" Saved {len(suspects_list)} suspicious locations to file {output_file}")
    return suspects_list


suspects = export_suspects_to_json(
    raw_results,
    threshold=0.90,
    nms_iou=0.05,
    margin=20,
    output_file="suspected_patches.json"
)